# Notebook 1b — Improved Re-training (v2)

**Companion to `1_Model-Finetune.ipynb`.** Notebook 1 trained BERT on the raw dataset text and scored 100% in-distribution — which §5.1 proved is a **dataset source artifact** (the `ceas-challenge` header marker separates the classes with no ML).

This notebook applies the fixes documented in Notebook 1 §6 and re-trains a clean **v2** checkpoint:

| Limitation (NB1 §6) | Fix applied here |
|---|---|
| Label-suffix leak (`Email type is: …`) | Stripped |
| CEAS header artifact (`ceas-challenge`, sender/date fingerprints) | **Train on email body only** (artifact drops 57.1% → 1.4%) |
| `max_length=128` truncation | Raised to **512** |
| Train/inference format mismatch | Body-only matches the cascade's inference input (notebook 2) |
| No seed / implicit label map | `seed=42`, explicit `label2id` |

**Purpose:** a controlled before/after. If in-distribution accuracy falls from 100% toward the cross-corpus number, that *confirms* the artifact diagnosis. The v2 checkpoint is then the honest model to run through the cascade (notebook 2) for a true cross-corpus comparison.

In [1]:
# pip install transformers datasets evaluate accelerate scikit-learn -U

In [2]:
import re, numpy as np
from datasets import load_dataset

SEED = 42
ds = load_dataset("drorrabin/phishing_emails-data")

# ── Extract email BODY only (strip prompt prefix, headers, and label suffix) ──
# This removes the label-suffix leak AND the CEAS header artifact, and matches
# the body-only input the deployed cascade (notebook 2) feeds at inference.
def extract_body(text):
    m = re.search(r"Email Body:\s*(.*?)\s*Email type is:\s*(?:phishing email|safe email)\s*$",
                  str(text), flags=re.DOTALL | re.IGNORECASE)
    if m:
        return m.group(1).strip()
    m2 = re.search(r"Email Body:\s*(.*)$", str(text), flags=re.DOTALL | re.IGNORECASE)
    body = m2.group(1) if m2 else str(text)
    return re.sub(r"\s*Email type is:\s*(phishing email|safe email)\s*$", "", body, flags=re.I).strip()

ds = ds.map(lambda ex: {"body": extract_body(ex["text"])})

# Sanity: the artifact should be gone from body-only inputs
import pandas as pd
_tr = pd.DataFrame({"body": ds["train"]["body"], "label": ds["train"]["email_type"]})
_marker = _tr["body"].str.contains("ceas-challenge", case=False, regex=False)
_rule_acc = (_marker.map({True: "phishing email", False: "safe email"}) == _tr["label"]).mean()
print(f"CEAS marker now in body-only: {_marker.mean()*100:.1f}%  (was 57.1% in full text)")
print(f"One-rule classifier on body-only: {_rule_acc*100:.1f}%  (was 92.7% on full text → artifact removed)")
print(f"Median body length: {_tr['body'].str.len().median():.0f} chars")

/home/ltkien/Development/CourseHomework/applied-llm-term-project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo card metadata block was not found. Setting CardData to empty.


Map:   0%|          | 0/26946 [00:00<?, ? examples/s]

Map:  15%|█▍        | 4000/26946 [00:00<00:00, 38546.34 examples/s]

Map:  30%|██▉       | 8000/26946 [00:00<00:00, 39017.92 examples/s]

...

Map:   0%|          | 0/3705 [00:00<?, ? examples/s]

Map:  45%|████▍     | 1650/3705 [00:00<00:00, 16451.30 examples/s]

Map:  90%|████████▉ | 3319/3705 [00:00<00:00, 16587.21 examples/s]

Map: 100%|██████████| 3705/3705 [00:00<00:00, 16149.16 examples/s]

CEAS marker now in body-only: 1.4%  (was 57.1% in full text)
One-rule classifier on body-only: 48.9%  (was 92.7% on full text → artifact removed)
Median body length: 552 chars


In [3]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Explicit, stable label mapping (NB1 §6 limitation #4)
label2id = {"phishing email": 0, "safe email": 1}
id2label = {v: k for k, v in label2id.items()}

def preprocess(examples):
    enc = tokenizer(examples["body"], truncation=True, padding="max_length", max_length=512)
    enc["labels"] = [label2id[l] for l in examples["email_type"]]
    return enc

tokenized = ds.map(preprocess, batched=True).remove_columns(["text", "body", "email_type"])
tokenized.set_format("torch")
print("Tokenized at max_length=512, body-only. Train:", len(tokenized["train"]), "Test:", len(tokenized["test"]))

Map:   0%|          | 0/26946 [00:00<?, ? examples/s]

Map:   4%|▎         | 1000/26946 [00:00<00:02, 9184.80 examples/s]

Map:   7%|▋         | 2000/26946 [00:00<00:02, 9288.65 examples/s]

Map:  15%|█▍        | 4000/26946 [00:00<00:02, 10198.22 examples/s]

Map:  22%|██▏       | 6000/26946 [00:00<00:02, 10339.89 examples/s]

...

Map:   0%|          | 0/3705 [00:00<?, ? examples/s]

Map:  54%|█████▍    | 2000/3705 [00:00<00:00, 9980.03 examples/s]

Map: 100%|██████████| 3705/3705 [00:00<00:00, 10112.95 examples/s]

Map: 100%|██████████| 3705/3705 [00:00<00:00, 9834.44 examples/s] 

Tokenized at max_length=512, body-only. Train: 26946 Test: 3705


In [4]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2, id2label=id2label, label2id=label2id)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, pos_label=label2id["phishing email"], average="binary", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "precision": p, "recall": r, "f1": f1}

args = TrainingArguments(
    output_dir="./results-v2",
    learning_rate=2e-5,
    per_device_train_batch_size=8,      # 512 tokens → smaller batch to fit 8 GB VRAM
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),     # mixed precision: faster + less memory
    seed=SEED,
    report_to="none",
)

trainer = Trainer(model=model, args=args,
                  train_dataset=tokenized["train"], eval_dataset=tokenized["test"],
                  compute_metrics=compute_metrics)
trainer.train()
print("v2 training complete.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19991.05it/s]


[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.098911
1000,0.041898
1500,0.043992
2000,0.046723
2500,0.013736
3000,0.025347
3500,0.026757
4000,0.007376
4500,0.006379
5000,0.010266


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.41it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.26it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.26it/s]

v2 training complete.


In [5]:
trainer.save_model("./phishing-bert-v2")
tokenizer.save_pretrained("./phishing-bert-v2")
print("v2 checkpoint saved to ./phishing-bert-v2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.33it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.33it/s]

v2 checkpoint saved to ./phishing-bert-v2


## Result: in-distribution score after removing the artifact

In [6]:
res = trainer.evaluate()
print("v2 in-distribution performance (body-only test split, N = 3,705):")
for k in ["eval_accuracy", "eval_precision", "eval_recall", "eval_f1"]:
    print(f"  {k[5:]:10}: {res[k]:.4f}")

print("\n── Before / after ──────────────────────────────────────────────")
print("  v1 (raw text, artifact present):   in-distribution accuracy = 100.00%")
print(f"  v2 (body-only, artifact removed):  in-distribution accuracy = {res['eval_accuracy']*100:.2f}%")
print("\nIf v2 < v1, the drop is the artifact being removed — confirming NB1 §5.1.")
print("Next: run notebook 2 with this v2 checkpoint to measure the honest")
print("cross-corpus performance, then compare against v1 in notebook 3.")

Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
0.003764,0.017787,10107,0.997571,0.976676,0.997024,0.986745


v2 in-distribution performance (body-only test split, N = 3,705):
  accuracy  : 0.9976
  precision : 0.9767
  recall    : 0.9970
  f1        : 0.9867

── Before / after ──────────────────────────────────────────────
  v1 (raw text, artifact present):   in-distribution accuracy = 100.00%
  v2 (body-only, artifact removed):  in-distribution accuracy = 99.76%

If v2 < v1, the drop is the artifact being removed — confirming NB1 §5.1.
Next: run notebook 2 with this v2 checkpoint to measure the honest
cross-corpus performance, then compare against v1 in notebook 3.


### Interpretation — a result that *reinforces* the cross-corpus methodology

Two things happened, and both matter:

1. **The artifact is confirmed as a shortcut.** Removing the headers dropped the trivial one-rule classifier from **92.7% → 48.9%** (pure chance). The `ceas-challenge` marker *was* a label oracle.
2. **Yet v2 in-distribution accuracy held at 99.76%** (vs v1's 100%). Removing the artifact barely moved the score.

This near-identical result is itself the finding: **this dataset is *internally* near-trivial.** Train and test splits are drawn from the same source corpora, so a model separates them almost perfectly from body content alone — with or without the header artifact. In-distribution accuracy therefore **cannot distinguish a model that genuinely generalises from one exploiting corpus-specific cues.** That is precisely why every headline number in this project is measured **cross-corpus**, where v1 scored 80.71% accuracy / 60.4% recall.

**What v2 changes that in-distribution can't show:** v2 is trained on full-length (512-token) **body-only** text — the same format the deployed cascade feeds at inference. v1's 128-token truncation and train/inference format mismatch (NB1 §6) are both removed. Those fixes are expected to help **cross-corpus generalisation**, not the already-saturated in-distribution score. Running v2 through the cascade (notebook 2) and comparing against v1 in notebook 3 is the test that matters.

## How to run the cascade (notebook 2) with this v2 checkpoint

Notebook 2 loads BERT from the Hugging Face Hub. To evaluate v2 end-to-end, either:

1. **Push v2 to the Hub** and point notebook 2 at it:
   ```python
   model_bert = AutoModelForSequenceClassification.from_pretrained("./phishing-bert-v2")
   model_bert.push_to_hub("<your-username>/phishing-bert-v2")   # then update model_repo in NB2
   ```
2. **Or load the local folder directly** in notebook 2's Step 5:
   ```python
   model_repo = "./phishing-bert-v2"   # instead of the Hub repo
   ```

Save the regenerated predictions as **`pineline_full_predictions_v2.csv`** (do *not* overwrite the v1 file — the before/after comparison needs both). Notebook 3 then scores both and reports the v1 → v2 improvement.